In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import  re
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
import joblib

In [2]:
df=pd.read_csv('P:\AI-Resume-Screener\data\dataset_cleaned.csv')

In [3]:
df.head

<bound method NDFrame.head of                              Role  \
0           E-commerce Specialist   
1                  Game Developer   
2      Human Resources Specialist   
3           E-commerce Specialist   
4           E-commerce Specialist   
...                           ...   
10169             Product Manager   
10170                 UI Engineer   
10171                 UI Engineer   
10172               Data Engineer   
10173             Product Manager   

                                              Transcript  \
0      Interviewer: Good morning, Jason. It's great t...   
1      Interview Scene\n\nA conference room with a ta...   
2      Interview Setting: A conference room in a medi...   
3      Here's a simulated professional interview for ...   
4      Here's the simulated interview:\n\nInterviewer...   
...                                                  ...   
10169  Here's a simulated interview for a Product Man...   
10170  **Interviewer:** Hi Grace, thank you f

In [4]:
def cleanResume(resumeText):
    resumeText = re.sub('http\S+\s*', ' ', resumeText)  # remove URLs
    resumeText = re.sub('RT|cc', ' ', resumeText)  # remove RT and cc
    resumeText = re.sub('#\S+', '', resumeText)  # remove hashtags
    resumeText = re.sub('@\S+', '  ', resumeText)  # remove mentions
    resumeText = re.sub('[%s]' % re.escape("""!"#$%&'()*+,-./:;<=>?@[\]^_`{|}~"""), ' ', resumeText)  # remove punctuations
    resumeText = re.sub(r'[^\x00-\x7f]',r' ', resumeText) 
    resumeText = re.sub('\s+', ' ', resumeText)  # remove extra whitespace
    return resumeText

In [5]:
df['Resume_cleaned'] = df['Resume'].apply(lambda x:cleanResume(x))
df['Job_Description_cleaned'] = df['Job_Description'].apply(lambda x:cleanResume(x))
df['Transcript_cleaned'] = df['Transcript'].apply(lambda x:cleanResume(x))
df.head()

,Role,Transcript,Resume,decision,Job_Description,Resume_cleaned,Job_Description_cleaned,Transcript_cleaned
0,E-commerce Specialist,"Interviewer: Good morning, Jason. It's great t...",Here's a professional resume for Jason Jones:\...,reject,Be part of a passionate team at the forefront ...,Here s a professional resume for Jason Jones J...,Be part of a passionate team at the forefront ...,Interviewer Good morning Jason It s great to m...
1,Game Developer,Interview Scene\n\nA conference room with a ta...,Here's a professional resume for Ann Marshall:...,select,Help us build the next-generation products as ...,Here s a professional resume for Ann Marshall ...,Help us build the next generation products as ...,Interview Scene A conference room with a table...
2,Human Resources Specialist,Interview Setting: A conference room in a medi...,Here's a professional resume for Patrick Mccla...,reject,We need a Human Resources Specialist to enhanc...,Here s a professional resume for Patrick M lai...,We need a Human Resources Specialist to enhanc...,Interview Setting A conference room in a mediu...
3,E-commerce Specialist,Here's a simulated professional interview for ...,Here's a professional resume for Patricia Gray...,select,Be part of a passionate team at the forefront ...,Here s a professional resume for Patricia Gray...,Be part of a passionate team at the forefront ...,Here s a simulated professional interview for ...
4,E-commerce Specialist,Here's the simulated interview:\n\nInterviewer...,Here's a professional resume for Amanda Gross:...,reject,We are looking for an experienced E-commerce S...,Here s a professional resume for Amanda Gross ...,We are looking for an experienced E commerce S...,Here s the simulated interview Interviewer Goo...


In [6]:
vectorizer = CountVectorizer(stop_words='english')
analyzer = vectorizer.build_analyzer()
df['Resume'] = df['Resume_cleaned'].apply(lambda x: ' '.join(analyzer(x)))
df['Job_Description'] = df['Job_Description_cleaned'].apply(lambda x: ' '.join(analyzer(x)))
df['Transcript'] = df['Transcript_cleaned'].apply(lambda x: ' '.join(analyzer(x)))
df.head()


,Role,Transcript,Resume,decision,Job_Description,Resume_cleaned,Job_Description_cleaned,Transcript_cleaned
0,E-commerce Specialist,interviewer good morning jason great meet welc...,professional resume jason jones jason jones co...,reject,passionate team forefront machine learning com...,Here s a professional resume for Jason Jones J...,Be part of a passionate team at the forefront ...,Interviewer Good morning Jason It s great to m...
1,Game Developer,interview scene conference room table chairs a...,professional resume ann marshall ann marshall ...,select,help build generation products game developer ...,Here s a professional resume for Ann Marshall ...,Help us build the next generation products as ...,Interview Scene A conference room with a table...
2,Human Resources Specialist,interview setting conference room medium sized...,professional resume patrick lain patrick lain ...,reject,need human resources specialist enhance team t...,Here s a professional resume for Patrick M lai...,We need a Human Resources Specialist to enhanc...,Interview Setting A conference room in a mediu...
3,E-commerce Specialist,simulated professional interview commerce spec...,professional resume patricia gray patricia gra...,select,passionate team forefront cloud computing comm...,Here s a professional resume for Patricia Gray...,Be part of a passionate team at the forefront ...,Here s a simulated professional interview for ...
4,E-commerce Specialist,simulated interview interviewer good morning a...,professional resume amanda gross amanda gross ...,reject,looking experienced commerce specialist join t...,Here s a professional resume for Amanda Gross ...,We are looking for an experienced E commerce S...,Here s the simulated interview Interviewer Goo...


In [7]:
df=df.drop(['Resume_cleaned','Job_Description_cleaned','Transcript_cleaned'],axis=1)

In [8]:
word_vectorizer = TfidfVectorizer(
    stop_words='english',
    max_features=5000,
     ngram_range=(1, 2))
combined = pd.concat([
    df["Resume"],
    df["Job_Description"],
    df['Transcript']
])
word_vectorizer.fit(combined)
resume_vectorized=word_vectorizer.transform(df["Resume"])
job_description_vectorized=word_vectorizer.transform(df["Job_Description"])
transcript_vectorized=word_vectorizer.transform(df["Transcript"])

In [9]:
def jaccard_similarity(text1, text2):
    set1 = set(text1.split())
    set2 = set(text2.split())
    intersection = set1.intersection(set2)
    union = set1.union(set2)
    return len(intersection) / len(union) if len(union) != 0 else 0

df['jaccard_resume_jd'] = df.apply(lambda row: jaccard_similarity(row['Resume'], row['Job_Description']), axis=1)
df['jaccard_resume_transcript'] = df.apply(lambda row: jaccard_similarity(row['Resume'], row['Transcript']), axis=1)
df['jaccard_jd_transcript'] = df.apply(lambda row: jaccard_similarity(row['Job_Description'], row['Transcript']), axis=1)

In [10]:

jd_resume_similarity = []
for i in range(len(df)):
    score = cosine_similarity(
        resume_vectorized[i],
        job_description_vectorized[i]
    )[0][0]

    jd_resume_similarity.append(score)

In [11]:

jd_transcript_similarity = []
for i in range(len(df)):
    score = cosine_similarity(
        transcript_vectorized[i],
        job_description_vectorized[i]
    )[0][0]

    jd_transcript_similarity.append(score)

In [12]:

resume_transcript_similarity = []
for i in range(len(df)):
    score = cosine_similarity(
        resume_vectorized[i],
        transcript_vectorized[i]
    )[0][0]

    resume_transcript_similarity.append(score)

In [13]:
df['jd_resume_similarity'] = jd_resume_similarity
df['jd_transcript_similarity'] = jd_transcript_similarity
df['resume_transcript_similarity'] = resume_transcript_similarity
df

,Role,Transcript,Resume,decision,Job_Description,jaccard_resume_jd,jaccard_resume_transcript,jaccard_jd_transcript,jd_resume_similarity,jd_transcript_similarity,resume_transcript_similarity
0,E-commerce Specialist,interviewer good morning jason great meet welc...,professional resume jason jones jason jones co...,reject,passionate team forefront machine learning com...,0.017544,0.070381,0.014286,0.110196,0.045556,0.212312
1,Game Developer,interview scene conference room table chairs a...,professional resume ann marshall ann marshall ...,select,help build generation products game developer ...,0.050000,0.046025,0.022222,0.157439,0.095865,0.117589
2,Human Resources Specialist,interview setting conference room medium sized...,professional resume patrick lain patrick lain ...,reject,need human resources specialist enhance team t...,0.028090,0.121813,0.030172,0.165360,0.043850,0.332292
3,E-commerce Specialist,simulated professional interview commerce spec...,professional resume patricia gray patricia gra...,select,passionate team forefront cloud computing comm...,0.020408,0.134897,0.011719,0.090639,0.068059,0.252923
4,E-commerce Specialist,simulated interview interviewer good morning a...,professional resume amanda gross amanda gross ...,reject,looking experienced commerce specialist join t...,0.018182,0.113565,0.019512,0.035783,0.055557,0.195964
...,...,...,...,...,...,...,...,...,...,...,...
10169,Product Manager,simulated interview product manager role candi...,sample resume diana miller diana miller contac...,reject,comprehensive job description product manager ...,0.186335,0.106007,0.101852,0.430270,0.168836,0.276525
10170,UI Engineer,interviewer hi grace thank coming today start ...,sample resume grace taylor grace taylor contac...,reject,sample job description ui engineer job title u...,0.228571,0.087209,0.084011,0.435852,0.091417,0.199382
10171,UI Engineer,simulated interview ui engineer role hank brow...,sample resume hank brown hank brown ui enginee...,select,job description ui engineer role job title ui ...,0.178977,0.138425,0.171569,0.399942,0.325333,0.331898
10172,Data Engineer,simulated interview data engineer role intervi...,sample resume diana wilson diana wilson contac...,reject,comprehensive job description data engineer ro...,0.200000,0.113879,0.128289,0.396570,0.311673,0.315879


In [14]:
df["resume_length"] = df["Resume"].apply(lambda x: len(x.split()))
df["jd_length"] = df["Job_Description"].apply(lambda x: len(x.split()))
df['Transcript_length'] = df['Transcript'].apply(lambda x: len(x.split()))

In [15]:
keyword_overlap_ratio=[]
stop_words = set(ENGLISH_STOP_WORDS)
for i in range(len(df)):
    job_words=df['Job_Description'][i].split(" ")
    resume_words=df['Resume'][i].split(" ")
    resume_words = {
        w for w in resume_words
        if w not in stop_words
    }

    jd_words = {
        w for w in job_words
        if w not in stop_words
    }
    comman=resume_words.intersection(jd_words)
    keyword_overlap_ratio.append(len(comman)/len(job_words))

df['keyword_overlap_ratio_resume_jd'] = keyword_overlap_ratio

In [16]:
keyword_overlap_ratio_transcript_jd=[]
stop_words = set(ENGLISH_STOP_WORDS)
for i in range(len(df)):
    job_words=df['Job_Description'][i].split(" ")
    transcript_words=df['Transcript'][i].split(" ")
    transcript_words = {
        w for w in transcript_words
        if w not in stop_words
    }

    jd_words = {
        w for w in job_words
        if w not in stop_words
    }
    comman=transcript_words.intersection(jd_words)
    keyword_overlap_ratio_transcript_jd.append(len(comman)/len(job_words))

df['keyword_overlap_ratio_transcript_jd'] = keyword_overlap_ratio_transcript_jd

In [17]:
df.to_csv('P:\AI-Resume-Screener\data\dataset_cleaned.csv',index=False)

In [18]:
df.head()

,Role,Transcript,Resume,decision,Job_Description,jaccard_resume_jd,jaccard_resume_transcript,jaccard_jd_transcript,jd_resume_similarity,jd_transcript_similarity,resume_transcript_similarity,resume_length,jd_length,Transcript_length,keyword_overlap_ratio_resume_jd,keyword_overlap_ratio_transcript_jd
0,E-commerce Specialist,interviewer good morning jason great meet welc...,professional resume jason jones jason jones co...,reject,passionate team forefront machine learning com...,0.017544,0.070381,0.014286,0.110196,0.045556,0.212312,253,11,318,0.272727,0.272727
1,Game Developer,interview scene conference room table chairs a...,professional resume ann marshall ann marshall ...,select,help build generation products game developer ...,0.050000,0.046025,0.022222,0.157439,0.095865,0.117589,40,11,331,0.181818,0.454545
2,Human Resources Specialist,interview setting conference room medium sized...,professional resume patrick lain patrick lain ...,reject,need human resources specialist enhance team t...,0.028090,0.121813,0.030172,0.165360,0.043850,0.332292,286,13,362,0.384615,0.538462
3,E-commerce Specialist,simulated professional interview commerce spec...,professional resume patricia gray patricia gra...,select,passionate team forefront cloud computing comm...,0.020408,0.134897,0.011719,0.090639,0.068059,0.252923,234,11,467,0.272727,0.272727
4,E-commerce Specialist,simulated interview interviewer good morning a...,professional resume amanda gross amanda gross ...,reject,looking experienced commerce specialist join t...,0.018182,0.113565,0.019512,0.035783,0.055557,0.195964,271,12,323,0.250000,0.333333


In [19]:
joblib.dump(word_vectorizer, r'P:\AI-Resume-Screener\models\word_vectorizer_transcript.pkl')

['P:\\AI-Resume-Screener\\models\\word_vectorizer_transcript.pkl']